In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Convert a number to hex and return a list of digit tokens
def number_to_hex_digits(num):
    hex_str = hex(num)[2:].upper()  # Convert to hex (remove '0x' and make uppercase)
    return [int(char, 16) for char in hex_str]  # Convert hex characters to indices

# Generate dataset: multiple numbers → hex → sorted hex list
def generate_data(num_samples=1000, max_num=999):
    data = []
    for _ in range(num_samples):
        numbers = [random.randint(0, max_num) for _ in range(random.randint(1, 5))]  # Random list of 1-5 numbers
        hex_digits = [digit for num in numbers for digit in number_to_hex_digits(num)]  # Flatten hex digits
        sorted_hex_digits = sorted(hex_digits)  # Sorting step
        data.append((hex_digits, sorted_hex_digits))
    return data

# Token and Positional Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

    def forward(self, x):
        return self.embedding(x)
    
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=20):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

    def forward(self, x):
        positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        return x + self.pos_embedding(positions)

# Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.norm2(x + ffn_output)

# Transformer Encoder Model
class TransformerSorter(nn.Module):
    def __init__(self, vocab_size=16, embed_dim=64, num_heads=4, hidden_dim=128, num_layers=4):
        super().__init__()
        self.token_embed = TokenEmbedding(vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim)
        self.encoder_layers = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.token_embed(x)
        x = self.positional_encoding(x)
        x = self.encoder_layers(x)
        return self.output_layer(x)

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 
LR = 0.001

# Model, Loss, Optimizer
model = TransformerSorter().to(device)  # Move model to GPU if available
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

# Generate Training Data
dataset = generate_data()
train_data = [(torch.tensor(x, device=device), torch.tensor(y, device=device)) for x, y in dataset]

scaler = torch.cuda.amp.GradScaler()

for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in train_data:
        x, y = x.unsqueeze(0), y.unsqueeze(0)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():  # Mixed precision
            output = model(x)
            loss = criterion(output.view(-1, 16), y.view(-1))
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_data):.4f}")

# Test Prediction Function
def predict_sorting(model, numbers):
    model.eval()
    with torch.no_grad():
        hex_seq = [digit for num in numbers for digit in number_to_hex_digits(num)]
        x = torch.tensor(hex_seq, device=device).unsqueeze(0)  # Add batch dim
        output = model(x)
        predicted = torch.argmax(output, dim=-1).squeeze(0)
        predicted_hex = [format(p.item(), 'X') for p in predicted]
        return predicted_hex



Training on: cuda
Epoch 1, Loss: 1.4072
Epoch 2, Loss: 0.8169
Epoch 3, Loss: 0.6097
Epoch 4, Loss: 0.5595
Epoch 5, Loss: 0.4479
Epoch 6, Loss: 0.4116
Epoch 7, Loss: 0.3821
Epoch 8, Loss: 0.3612
Epoch 9, Loss: 0.2834
Epoch 10, Loss: 0.3224
Epoch 11, Loss: 0.2861
Epoch 12, Loss: 0.2756
Epoch 13, Loss: 0.2614
Epoch 14, Loss: 0.2358
Epoch 15, Loss: 0.2509
Epoch 16, Loss: 0.2235
Epoch 17, Loss: 0.1850
Epoch 18, Loss: 0.2309
Epoch 19, Loss: 0.2136
Epoch 20, Loss: 0.1794
Epoch 21, Loss: 0.1701
Epoch 22, Loss: 0.1617
Epoch 23, Loss: 0.2255
Epoch 24, Loss: 0.1455
Epoch 25, Loss: 0.1626
Epoch 26, Loss: 0.1919
Epoch 27, Loss: 0.1502
Epoch 28, Loss: 0.1349
Epoch 29, Loss: 0.1462
Epoch 30, Loss: 0.1685
Epoch 31, Loss: 0.1306
Epoch 32, Loss: 0.1420
Epoch 33, Loss: 0.1072
Epoch 34, Loss: 0.2063
Epoch 35, Loss: 0.0857
Epoch 36, Loss: 0.1163
Epoch 37, Loss: 0.1282
Epoch 38, Loss: 0.1346
Epoch 39, Loss: 0.1134
Epoch 40, Loss: 0.1026
Epoch 41, Loss: 0.1201
Epoch 42, Loss: 0.1491
Epoch 43, Loss: 0.0931
Ep

In [10]:
numbers = [25, 8, 160]  # Example list of numbers
sorted_hex = predict_sorting(model, numbers)
print(f"Original Numbers: {numbers} → Sorted Hex Digits: {sorted_hex}")

Original Numbers: [25, 8, 160] → Sorted Hex Digits: ['0', '1', '8', '9', 'A']
